# Build surface
###### Last updated 2024-04-24
This notebook walks though constructing a surface using a single long chain of bead type "A". This can be used to build homogenous surfaces in PIMMS.

### Approach
Broadly, the approach here is to:

1. Build a restart file where a single chain snakes along the bottom of the simulation box.
2. Save that dictionary as a restart file called `new_restart.pimms` (note this could be called literally anything but the `.pimms` extension seems sufficiently unique!)
3. Define a freezefile (`freezefile.in`) which freezes that one long 'surface' chain (which will be chainID 1 because it's the first and only chain in the restart file.
4. In the associated keyfile, run a simulation where we read in that restart file and freeze file.
5. Profit (?).

All the prep work is done in this directory, so you should be able to simply run

    pimms -k KEYFILE.kf

And a simulations with a surface will run. NOTE the surface is very large, so to make this computationally tractable we have the `SAVE_AT_END` keyword set to True, otherwise writing out to XTC file every few steps becomes very expensive. As it stands on an M3 Macbook Pro this simulation should take ~4-5 minutes to run.    

In [1]:
# import pickle!
import pickle

In [2]:
# b is going to end up being the ou
b = {}

# n defines the lattice size
n = 60 

# pos will be a list of 3-position lists defining [x,y,z] positions of
# each bead on the bottom surface
pos = []

# Start filling the grid, snaking up and down
count = 0
for col in range(n):
    if col % 2 == 0:
        
        # even columns go up
        for row in range(n):            
            pos.append([row,col,0])            
    else:
        # odd columns go down
        for row in range(n-1, -1, -1):            
            pos.append([row,col,0])      

In [3]:
# now we define the required components of PIMMS restart file

## define 1 chain (REMEMBER chainID must start at 1!) N
# pos = positions
# 'A'*len(pos) = string of As that describes the 'sequence'
# 0 = the chainType ID which should be the same for chains of the same sequence (although strictly speaking does not have to be...)

b['CHAINS'] = {}
b['CHAINS'][1] = [pos, 'A'*len(pos), 0]

# box dimensions
b['DIMENSIONS'] = [n,n,n]

# energy but this does not actually matter and is not actually used
b['ENERGY'] = 0

# defines mode of simulation
b['HARDWALL'] = True


In [4]:
# write restart file out
with open('new_restart.pimms', 'wb') as handle:
    pickle.dump(b, handle)
